In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [ ]:
from huggingface_hub import login

# 2. 복사한 토큰을 여기에 붙여넣으세요 (보안을 위해 직접 입력창에 넣는 것을 권장)
# 빈칸으로 실행하면 토큰을 입력하라는 텍스트 박스가 뜹니다.
login()


In [3]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers import utils

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['PYTHONIOENCODING'] = 'utf-8' # 파이썬 입출력 인코딩 강제
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

print("PyTorch 버전:", torch.__version__)
print("GPU(CUDA) 사용 가능 여부:", torch.cuda.is_available())

PyTorch 버전: 2.5.1+cu121
GPU(CUDA) 사용 가능 여부: True


In [4]:
import json
import os
from tqdm.notebook import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ==========================================
# 2. 2060 Super(VRAM 8GB)용 4비트 양자화 설정
# ==========================================
model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)


# ==========================================
# 3. 모델 및 토크나이저 로드 (메모리 적재)
# ==========================================
print("-> 4비트 압축 모드로 Qwen 모델을 VRAM에 로드합니다...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,  # ◀여기에 4비트 설정이 들어갑니다!
    device_map="auto"
)

print("-> 로드 완료! 2060 Super에서 구동 준비가 끝났습니다.\n")

# ==========================================
# 2. 진짜 내 JSON 데이터로 오픈북(지식 베이스) 구축
# ==========================================
print("-> 2. JSON 지식 파일 읽기 및 벡터 DB 생성 중...")

json_path = "./data/rag_data3/final_augmented_merged_rag_data3.json"

with open(json_path, 'r', encoding='utf-8') as f:
    rag_data = json.load(f)

raw_texts = []


C:\Users\zin\AppData\Local\Temp\ipykernel_23680\3804956311.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


-> 4비트 압축 모드로 Qwen 모델을 VRAM에 로드합니다...


Exception in thread Thread-8 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\zin\anaconda3\envs\p312\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "c:\Users\zin\anaconda3\envs\p312\Lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\zin\anaconda3\envs\p312\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\zin\anaconda3\envs\p312\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 6: invalid start byte


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

-> 로드 완료! 2060 Super에서 구동 준비가 끝났습니다.

-> 2. JSON 지식 파일 읽기 및 벡터 DB 생성 중...


In [5]:

if isinstance(rag_data, list):
    for item in rag_data:
        if isinstance(item, str):
            raw_texts.append(item)
        elif isinstance(item, dict):
            content = item.get('context') or item.get('text') or item.get('body') or str(item)
            raw_texts.append(content)
elif isinstance(rag_data, dict):
    raw_texts = [str(v) for v in rag_data.values()]

print(f"-> 총 {len(raw_texts)}개의 데이터 조각을 로드했습니다.")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)

split_docs = text_splitter.create_documents(raw_texts)
print(f"-> 가위질 완료! 총 {len(split_docs)}개의 청크(Chunk)로 분할됨.")




-> 총 5563개의 데이터 조각을 로드했습니다.
-> 가위질 완료! 총 5563개의 청크(Chunk)로 분할됨.


In [6]:
# 임베딩 모델 로드 및 벡터 DB 구축
embedding_model = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-base")
vector_db = FAISS.from_documents(split_docs, embedding_model)

# (선택) 빌드하면 오래 걸리므로, 내 컴퓨터에 'my_faiss_index' 폴더로 저장
vector_db.save_local("my_faiss_index")
print("-> [성공] 진짜 나만의 벡터 데이터베이스 구축 및 로컬 저장 완료!\n")


'''print("-> ④ 한국어 임베딩 모델 로드 중...")
embedding_model = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-base")

print("-> ⑤ 벡터 변환 및 DB 빌드 시작 (tqdm 진행바 활성화)...")

# 1. 먼저 첫 번째 문서 조각 하나만 가지고 기준이 될 벡터 DB를 생성합니다.
vector_db = FAISS.from_documents([split_docs[0]], embedding_model)

# 2. 나머지 문서 조각들을 100개씩 묶어서 tqdm 진행 바를 보면서 추가합니다.
batch_size = 100  
for i in tqdm(range(1, len(split_docs), batch_size), desc="벡터 DB 빌드 중"):
    batch = split_docs[i:i+batch_size]
    vector_db.add_documents(batch)  # 문장들을 숫자로 변환하며 누적 저장

# 3. 연산이 다 끝나면 로컬 하드디스크에 저장
vector_db.save_local("my_faiss_index")
print("-> [성공] 진짜 나만의 벡터 데이터베이스 구축 및 로컬 저장 완료!\n")'''


C:\Users\zin\AppData\Local\Temp\ipykernel_23680\110546147.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-base")


'print("-> ④ 한국어 임베딩 모델 로드 중...")\nembedding_model = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-base")\n\nprint("-> ⑤ 벡터 변환 및 DB 빌드 시작 (tqdm 진행바 활성화)...")\n\n# 1. 먼저 첫 번째 문서 조각 하나만 가지고 기준이 될 벡터 DB를 생성합니다.\nvector_db = FAISS.from_documents([split_docs[0]], embedding_model)\n\n# 2. 나머지 문서 조각들을 100개씩 묶어서 tqdm 진행 바를 보면서 추가합니다.\nbatch_size = 100  \nfor i in tqdm(range(1, len(split_docs), batch_size), desc="벡터 DB 빌드 중"):\n    batch = split_docs[i:i+batch_size]\n    vector_db.add_documents(batch)  # 문장들을 숫자로 변환하며 누적 저장\n\n# 3. 연산이 다 끝나면 로컬 하드디스크에 저장\nvector_db.save_local("my_faiss_index")\nprint("-> [성공] 진짜 나만의 벡터 데이터베이스 구축 및 로컬 저장 완료!\n")'

In [ ]:
# ==========================================
# 3. RAG 기반 질의응답(QA) 핵심 함수
# ==========================================
def run_rag_qa(user_question):
    print("\n" + "="*50)
    print(f"[유저 질문]: {user_question}")
    print("="*50)
    
    related_docs = vector_db.similarity_search(user_question, k=3)
    
    context = "\n---\n".join([doc.page_content for doc in related_docs])
    
    #print(f"🔍 [벡터 DB에서 찾아온 관련 지식 top-3]:\n{context}\n")
    print("Qwen이 답변을 생성 중입니다...")
    

    system_instruction = (
        "You are a helpful assistant. "
        "반드시 주어진 [참고 문서]의 내용에만 기반하여 질문에 정확하고 정직하게 답변하세요. "
        "문서에 나와있지 않은 애매한 정보는 절대 지어내지 마세요."
    )
    
    user_content = f"[참고 문서]\n{context}\n\n[질문]\n{user_question}"
    
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_content}
    ]
    
    # ③ 모델 입력 및 텍스트 생성
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=512, 
        temperature=0.1,     
        top_p=0.9
    )
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response


# ==========================================
# 4. 내 JSON 데이터를 기반으로 실전 QA 테스트
# ==========================================
my_test_question = "선익시스템으로 상호명 변경 시기는?"

final_answer = run_rag_qa(my_test_question)
print(f"\n[Qwen의 RAG 기반 최종 답변]:\n{final_answer}")


[유저 질문]: 선익시스템으로 상호명 변경 시기는?
🤖 Qwen이 답변을 생성 중입니다...

[Qwen의 RAG 기반 최종 답변]:
선익시스템으로 상호명 변경 시기는 1998년 4월입니다.


In [8]:
import textwrap
my_test_question = "인재상은?"

final_answer = run_rag_qa(my_test_question)
wrapped_answer = textwrap.fill(final_answer, width=50)
print(f"\n[Qwen의 RAG 기반 최종 답변]:\n{wrapped_answer}")

my_test_question = "Sunic system股票的交易资金总额是多少？"

final_answer = run_rag_qa(my_test_question)
wrapped_answer = textwrap.fill(final_answer, width=50)
print(f"\n[Qwen의 RAG 기반 최종 답변]:\n{wrapped_answer}")


[유저 질문]: 인재상은?
🤖 Qwen이 답변을 생성 중입니다...

[Qwen의 RAG 기반 최종 답변]:
선익시스템이 제시하는 인재상은 Specialty, Unconventionality,
Nice, Integrity, Challenge의 총 5가지 항목으로 구성되어 있습니다.

[유저 질문]: Sunic system股票的交易资金总额是多少？
🤖 Qwen이 답변을 생성 중입니다...

[Qwen의 RAG 기반 최종 답변]:
Sunic system股票的交易资金总额为6,558,165,000。
